# 03 Targeted Augmentation

## Objective
Perform targeted data augmentation on underrepresented crop disease classes to increase minority sample counts and boost class representation before preprocessing.

**Prerequisites:** `01_data_check.ipynb` and `02_dataset_visualization.ipynb`
**Next Step:** `04_data_preprocessing.ipynb`

### Import Libraries
Import required packages including OpenCV, Keras ImageDataGenerator, and NumPy.

In [2]:
import os
import cv2
import random
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("Libraries Loaded")


Libraries Loaded


### Set Dataset Path and Augmentation Target
Define the directory containing dataset subfolders and target minimum count of images per class.

In [3]:
DATASET_PATH = r"/content/drive/MyDrive/New_cnn_models_phd/cnn_model/Final_images_phd"

TARGET_IMAGES = 40

print("Target images per class:", TARGET_IMAGES)

Target images per class: 40


### Define Keras Image Augmentor
Configure ImageDataGenerator with rotation, shift, zoom, brightness, and horizontal flip properties.

In [4]:
augmentor = ImageDataGenerator(

    rotation_range=20,

    width_shift_range=0.10,

    height_shift_range=0.10,

    zoom_range=0.10,

    brightness_range=[0.8, 1.2],

    horizontal_flip=True,

    fill_mode='nearest'
)

print("Augmentor Ready")

Augmentor Ready


### Scan Class Folders
Search the dataset directory recursively to identify all valid crop/disease folder directories containing images.

In [5]:
image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
)

class_folders = []

for root, dirs, files in os.walk(DATASET_PATH):

    images = [
        f for f in files
        if f.lower().endswith(image_extensions)
    ]

    if len(images) > 0:

        class_folders.append(root)

print("Total Class Folders:", len(class_folders))

Total Class Folders: 0


### Attempt Targeted Augmentation using Keras ImageDataGenerator
Iterate through class folders and generate augmented images to meet the target count of 40 using Keras flow interface.

In [6]:
for folder in class_folders:

    images = [

        f for f in os.listdir(folder)

        if f.lower().endswith(image_extensions)

    ]

    current_count = len(images)

    if current_count >= TARGET_IMAGES:

        continue

    print("\n" + "="*60)
    print("Class:", os.path.basename(folder))
    print("Current:", current_count)
    print("Target :", TARGET_IMAGES)

    needed = TARGET_IMAGES - current_count

    generated = 0

    while generated < needed:

        image_name = random.choice(images)

        image_path = os.path.join(
            folder,
            image_name
        )

        image = cv2.imread(image_path)

        if image is None:
            continue

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        image = np.expand_dims(
            image,
            axis=0
        )

        aug_iter = augmentor.flow(
            image,
            batch_size=1
        )

        aug_img = next(aug_iter)[0].astype(np.uint8)

        save_name = (
            f"aug_{generated+1:03d}.jpg"
        )

        save_path = os.path.join(
            folder,
            save_name
        )

        cv2.imwrite(
            save_path,
            cv2.cvtColor(
                aug_img,
                cv2.COLOR_RGB2BGR
            )
        )

        generated += 1

    print(
        f"Generated {generated} images"
    )

### Perform Targeted Augmentation using OpenCV (Rotation & Zoom)
Use custom OpenCV operations (rotation and crop zoom) to augment and write images directly to files, ensuring all underrepresented folders reach 40 images.

In [7]:
import os
import cv2
import random

TARGET_IMAGES = 40

image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff"
)

for root, dirs, files in os.walk(DATASET_PATH):

    images = [
        f for f in files
        if f.lower().endswith(image_extensions)
    ]

    current_count = len(images)

    if current_count >= TARGET_IMAGES:
        continue

    # Skip if no images are found in the current directory
    if current_count == 0:
        print(f"\nSkipping: {os.path.basename(root)} (No images found)")
        continue

    print(f"\nProcessing: {os.path.basename(root)}")
    print(f"Current: {current_count}")

    needed = TARGET_IMAGES - current_count

    counter = 1

    while counter <= needed:

        img_name = random.choice(images)

        img_path = os.path.join(root, img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        h, w = img.shape[:2]

        # Rotation
        angle = random.choice([-15, -10, 10, 15])

        M = cv2.getRotationMatrix2D(
            (w//2, h//2),
            angle,
            1.0
        )

        aug_img = cv2.warpAffine(
            img,
            M,
            (w, h)
        )

        save_name = f"aug_rot_{counter:03d}.jpg"

        cv2.imwrite(
            os.path.join(root, save_name),
            aug_img
        )

        counter += 1

        if counter > needed:
            break

        # Slight Zoom
        scale = random.uniform(1.05, 1.10)

        zoom_img = cv2.resize(
            img,
            None,
            fx=scale,
            fy=scale
        )

        zh, zw = zoom_img.shape[:2]

        startx = (zw - w)//2
        starty = (zh - h)//2

        zoom_img = zoom_img[
            starty:starty+h,
            startx:startx+w
        ]

        save_name = f"aug_zoom_{counter:03d}.jpg"

        cv2.imwrite(
            os.path.join(root, save_name),
            zoom_img
        )

        counter += 1

    print(f"Final Count: {TARGET_IMAGES}")

### Verify Final Image Counts
Print the final count of images in each class folder to verify target of 40 was successfully met.

In [16]:
print("\nFINAL COUNTS\n")

for folder in class_folders:

    images = [

        f for f in os.listdir(folder)

        if f.lower().endswith(image_extensions)

    ]

    print(
        f"{os.path.basename(folder):35s} {len(images)}"
    )


FINAL COUNTS

Unhealthy_Guava                     40
algal_leaf_spot                     40
anthrecnose                         40
Fungal infection                    40
Nitrogen                            40
Magnesium                           40
Rust_fungal                         40
phosphorus                          40
pottasium                           40
Healthy_Guava                       51
Unhealthy_brinjal                   40
Little_leaf                         40
Magnessium                          40
Early_Leaf_spot                     63
Cercospora                          40
Leaf_curl                           40
Mosaic                              40
pest_damage                         53
Nitrogen                            40
Pottasium                           40
Eggplant Wilt Disease               59
Healthy_brinjal                     58
Unhealthy_castor                    40
Alternia Leaf Blight                41
Bacterial Leaf Blight               40
Cercospera

In [ ]:
print("\n" + "="*70)
print("✓ TARGETED AUGMENTATION COMPLETE")
print("="*70)
print("\n→ Next Step: 04_data_preprocessing.ipynb")
